In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from IPython.display import display


In [ ]:
#params
DOMAIN_LOW = -4.0
DOMAIN_HIGH = 4.0
N_TUNE = 1000
N_REPEATS_TUNE = 20
MAX_EPOCHS_TUNE = 1000
LR_FIXE_TUNE = 0.005
LAMBDA_DICT = {
    "Both auto-diff": [10.0, 12.5, 15.0, 16.0, 17.0, 17.5, 18.0, 19.0, 20.0, 22.5],
    "Both multitask": [0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0, 1.25, 1.5]
}

In [ ]:
def generate_combined_data(n_points, low=-4.0, high=4.0):
    X_raw = torch.rand(n_points, 2) * (high - low) + low
    X_norm = X_raw / 4.0

    y_raw_class = himmel(X_raw[:, 0], X_raw[:, 1]).unsqueeze(1)
    df_dx, df_dy = himmel_grad(X_raw[:, 0], X_raw[:, 1])
    y_raw_grad = torch.stack([df_dx, df_dy], dim=1)

    search_space = torch.linspace(low, high, 1000)
    grid_x, grid_y = torch.meshgrid(search_space, search_space, indexing='xy')
    max_val_class = torch.max(himmel(grid_x, grid_y)).item()
    grad_x_grid, grad_y_grid = himmel_grad(grid_x, grid_y)
    max_val_grad = torch.max(torch.abs(torch.stack([grad_x_grid, grad_y_grid]))).item()

    y_norm_class = y_raw_class / max_val_class
    y_norm_grad = y_raw_grad / max_val_grad

    if torch.cuda.is_available():
        return X_norm.cuda(), y_norm_class.cuda(), y_norm_grad.cuda(), max_val_class, max_val_grad
    return X_norm, y_norm_class, y_norm_grad, max_val_class, max_val_grad


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mae_criterion = nn.L1Loss()
mse_criterion = nn.MSELoss()

print(f"Début du Grand Fine-Tuning de PRÉCISION sur N={N_TUNE} ")

#préparation data
X_train, y_train_class, y_train_grad, MAX_VAL_CLASS, MAX_VAL_GRAD = generate_combined_data(N_TUNE, DOMAIN_LOW, DOMAIN_HIGH)
train_dataset = TensorDataset(X_train, y_train_class, y_train_grad)
train_loader = DataLoader(train_dataset, batch_size=250, shuffle=True)

X_val, y_val_class, y_val_grad, _, _ = generate_combined_data(2000, DOMAIN_LOW, DOMAIN_HIGH)
X_val_req = X_val.clone().detach().requires_grad_(True)

model_classes = {
    "Both auto-diff": HimmelblauNet4,
    "Both multitask": HimmelblauNet4_both,
}
#Finetunning
results_tune = []

for run_type in ["Both auto-diff", "Both multitask"]:
    print(f"\nAffinage de la stratégie : {run_type}")

    for lmbda in tqdm(LAMBDA_DICT[run_type], desc=f"Test des Lambdas"):
        val_losses = []

        for rep in tqdm(range(N_REPEATS_TUNE), desc=f"λ={lmbda}", leave=False):
            model = model_classes[run_type](nb_neurones=16).to(device)
            optimizer = optim.Adam(model.parameters(), lr=LR_FIXE_TUNE)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.8, patience=20, min_lr=1e-6)

            early_stopping = EarlyStopping(patience=80, path=f"temp_tune.pt")

            for epoch in range(MAX_EPOCHS_TUNE):
                model.train()
                for X_batch, y_class_batch, y_grad_batch in train_loader:
                    optimizer.zero_grad()
                    X_batch_req = X_batch.clone().detach().requires_grad_(True)

                    if run_type == "Both auto-diff":
                        preds_energy = model(X_batch_req)
                        preds_grad = torch.autograd.grad(
                            outputs=preds_energy, inputs=X_batch_req, grad_outputs=torch.ones_like(preds_energy), create_graph=True
                        )[0]
                        scaled_preds_grad = preds_grad * (MAX_VAL_CLASS / 4.0) / MAX_VAL_GRAD
                        loss = mse_criterion(preds_energy, y_class_batch) + lmbda * mse_criterion(scaled_preds_grad, y_grad_batch)

                    elif run_type == "Both multitask":
                        preds_energy, preds_grad = model(X_batch_req)
                        loss = mse_criterion(preds_energy, y_class_batch) + lmbda * mse_criterion(preds_grad, y_grad_batch)

                    loss.backward()
                    optimizer.step()

                #Eval
                model.eval()
                with torch.set_grad_enabled(True):
                    if run_type == "Both auto-diff":
                        preds_val = model(X_val_req)
                        grad_val = torch.autograd.grad(outputs=preds_val, inputs=X_val_req, grad_outputs=torch.ones_like(preds_val), create_graph=False)[0]
                        scaled_grad_val = grad_val * (MAX_VAL_CLASS / 4.0) / MAX_VAL_GRAD
                        val_loss = (mae_criterion(preds_val, y_val_class) + mae_criterion(scaled_grad_val, y_val_grad)).item()

                    elif run_type == "Both multitask":
                        preds_val, grad_val = model(X_val_req)
                        val_loss = (mae_criterion(preds_val, y_val_class) + mae_criterion(grad_val, y_val_grad)).item()

                scheduler.step(val_loss)
                early_stopping(val_loss, model)
                if early_stopping.early_stop:
                    break

            val_losses.append(early_stopping.best_loss)

        mean_val_loss = np.mean(val_losses)
        std_val_loss = np.std(val_losses)

        results_tune.append({
            "Stratégie": run_type,
            "Lambda (λ)": lmbda,
            "Erreur Moyenne": mean_val_loss,
            "Écart-Type": std_val_loss
        })


df_tune = pd.DataFrame(results_tune)

df_tune.to_csv("Resultats_Fine_Tuning_Lambdas.csv", index=False)

print("\nRÉSULTATS DU GRAND FINE-TUNING DE PRÉCISION :")
display(df_tune)

best_autodiff = df_tune[df_tune["Stratégie"] == "Both auto-diff"].sort_values("Erreur Moyenne").iloc[0]
best_multitask = df_tune[df_tune["Stratégie"] == "Both multitask"].sort_values("Erreur Moyenne").iloc[0]

print("\n")
print(f"VAINQUEUR 'Both auto-diff' : λ = {best_autodiff['Lambda (λ)']} (Erreur: {best_autodiff['Erreur Moyenne']:.5f} ± {best_autodiff['Écart-Type']:.5f})")
print(f"VAINQUEUR 'Both multitask' : λ = {best_multitask['Lambda (λ)']} (Erreur: {best_multitask['Erreur Moyenne']:.5f} ± {best_multitask['Écart-Type']:.5f})")